# 유튜브 데이터 추출

In [ ]:
# !pip install google-api-python-client pandas openpyxl -q

In [ ]:
import os

import re
import time
import pandas as pd
from googleapiclient.discovery import build


# =========================
# 1. API 키 입력
# =========================
API_KEY = os.environ["YOUTUBE_API_KEY"]

youtube = build("youtube", "v3", developerKey=API_KEY)


# =========================
# 2. 수집할 유튜브 영상 URL 입력
# =========================
video_urls = [
    "https://www.youtube.com/watch?v=JUafhCKFfII",
    "https://www.youtube.com/watch?v=wgz8Lv5ritw",
    "https://www.youtube.com/watch?v=6ZH0ytd4tO0",
    "https://www.youtube.com/watch?v=9t8WBI2djVg",
    "https://www.youtube.com/watch?v=MlVO3SeOrXI",
    "https://www.youtube.com/watch?v=C7lVqSaD2jI",
    "https://www.youtube.com/watch?v=wudvjc1odj8",
    "https://www.youtube.com/watch?v=_R3oFMG5BAU",
    "https://www.youtube.com/watch?v=N_F68ZEwmDQ",
    "https://www.youtube.com/watch?v=SCpGX8kCYeA"

]


# =========================
# 3. 영상 ID 추출 함수
# =========================
def get_video_id(url):
    patterns = [
        r"v=([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)",
        r"embed/([^?&]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    return None


# =========================
# 4. 댓글 최소 전처리 함수
# =========================
def clean_comment(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # HTML 태그 제거
    text = re.sub(r"<.*?>", " ", text)

    # URL 제거
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 줄바꿈 제거
    text = text.replace("\n", " ").replace("\r", " ")

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================
# 5. 댓글 수집
# =========================
all_comments = []
seen_comment_ids = set()

# 비정상 무한 수집 방지용 안전장치
MAX_COMMENTS_PER_VIDEO = 5000

print("수집할 영상 개수:", len(video_urls))
print("=" * 60)

for idx, url in enumerate(video_urls, start=1):
    video_id = get_video_id(url)

    if video_id is None:
        print(f"[스킵] 영상 ID 추출 실패: {url}")
        continue

    print(f"\n[{idx}/{len(video_urls)}] 댓글 수집 시작: {video_id}")

    page_count = 0
    video_comment_count = 0
    next_page_token = None

    try:
        while True:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                textFormat="plainText",
                order="relevance",
                pageToken=next_page_token
            )

            response = request.execute()
            page_count += 1

            items = response.get("items", [])
            new_count_this_page = 0

            for item in items:
                comment = item["snippet"]["topLevelComment"]
                comment_id = comment["id"]

                # 이미 수집한 댓글이면 스킵
                if comment_id in seen_comment_ids:
                    continue

                seen_comment_ids.add(comment_id)

                comment_data = comment["snippet"]

                original_comment = comment_data.get("textDisplay", "")
                clean_text = clean_comment(original_comment)

                all_comments.append({
                    "comment_id": comment_id,
                    "video_id": video_id,
                    "video_url": url,
                    "author": comment_data.get("authorDisplayName"),
                    "comment_original": original_comment,
                    "comment_clean": clean_text,
                    "like_count": comment_data.get("likeCount"),
                    "published_at": comment_data.get("publishedAt")
                })

                video_comment_count += 1
                new_count_this_page += 1

            print(
                f"[진행중] 영상 {video_id} | "
                f"{page_count}페이지 완료 | "
                f"이번 페이지 신규 {new_count_this_page}개 | "
                f"현재 {video_comment_count}개 수집"
            )

            # 이번 페이지에서 새 댓글이 하나도 없으면 반복 오류 가능성 있음
            if new_count_this_page == 0:
                print(f"[중단] 영상 {video_id} | 새 댓글이 0개라 반복 수집 방지 중단")
                break

            # 안전장치: 말도 안 되게 많이 수집되면 중단
            if video_comment_count >= MAX_COMMENTS_PER_VIDEO:
                print(f"[중단] 영상 {video_id} | {MAX_COMMENTS_PER_VIDEO}개 도달, 비정상 반복 방지")
                break

            next_page_token = response.get("nextPageToken")

            # 다음 페이지 없으면 정상 종료
            if not next_page_token:
                break

            time.sleep(0.1)

        print(f"[완료] 영상 {video_id} | 총 {video_comment_count}개 수집")

    except Exception as e:
        print(f"[에러 발생] 영상 {video_id}")
        print(e)
        continue


# =========================
# 6. 데이터프레임 생성 및 저장
# =========================
df = pd.DataFrame(all_comments)

print("\n" + "=" * 60)
print("전체 원본 수집 댓글 수:", len(df))

if len(df) > 0:
    df = df[df["comment_clean"].str.len() > 0].copy()

    # 혹시 모를 중복 최종 제거
    df = df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)

    print("중복 제거 후 댓글 수:", len(df))

    df.to_csv("youtube_comments_clean.csv", index=False, encoding="utf-8-sig")
    df.to_excel("youtube_comments_clean.xlsx", index=False)

    print("저장 완료")
    print("CSV 파일: youtube_comments_clean.csv")
    print("Excel 파일: youtube_comments_clean.xlsx")

else:
    print("수집된 댓글이 없습니다.")


# =========================
# 7. 확인
# =========================
df.head()

# GPT API 1차 감정분석

In [ ]:
# !pip install openai tqdm

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

# =========================
# 1. 댓글 파일 불러오기
# =========================
df = pd.read_csv("youtube_comments_clean.csv")

print("전체 댓글 수:", len(df))
df.head()

In [ ]:
import json
import time
import pandas as pd
from tqdm import tqdm

# =========================
# 1. 배치 감정분석 함수
# =========================

def analyze_sentiment_batch(comments, start_idx=0):
    comment_items = [
        {
            "id": start_idx + i,
            "comment": comment
        }
        for i, comment in enumerate(comments)
    ]

    prompt = f"""
너는 한국어 축구 팬덤 댓글을 분석하는 감정분석 분류기다.
분석 대상은 EPL 토트넘 관련 유튜브 댓글이다.

목표는 각 댓글이 '토트넘 관점에서' positive, negative, neutral 중 무엇인지 분류하는 것이다.

[핵심 기준]
- 토트넘 팀, 선수, 감독, 전술, 경기력, 결과를 칭찬하면 positive
- 토트넘 팀, 선수, 감독, 전술, 경기력, 결과를 비판/조롱/실망하면 negative
- 대상이 토트넘이 아니거나 불명확하면 neutral
- 출연진, 유튜버, 방송 자체에 대한 칭찬은 neutral
- 상대팀 칭찬은 보통 neutral
- 단, 상대팀 칭찬이 토트넘의 부진/패배를 강조하면 negative
- 비꼼, 조롱, 반어 표현은 실제 의미 기준으로 판단
- 확실하지 않으면 neutral

[축구 팬덤 표현]
- "미쳤다", "지린다", "찢었다", "월클", "GOAT"는 칭찬이면 positive
- "개못한다", "답답하다", "노답", "터졌다", "망했다"는 비판이면 negative
- "ㅋㅋ", "ㄹㅇ", "ㅇㅈ"만으로는 감정 판단이 어려우면 neutral

[target 기준]
- tottenham_team
- tottenham_player
- tottenham_manager
- opponent
- referee_or_league
- broadcast
- unclear

반드시 아래 JSON 배열 형식으로만 답하라.
설명, 마크다운, 코드블록은 쓰지 마라.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": "tottenham_team 또는 tottenham_player 또는 tottenham_manager 또는 opponent 또는 referee_or_league 또는 broadcast 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[분석할 댓글 목록]
{json.dumps(comment_items, ensure_ascii=False)}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 댓글 감정분석 전문가다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        results = json.loads(result_text)

        return results

    except Exception as e:
        print("배치 처리 에러:", e)

        return [
            {
                "id": start_idx + i,
                "sentiment": "error",
                "confidence": 0.0,
                "target": "error",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]


전체 댓글 실행

In [ ]:
# =========================
# 2. 전체 데이터 배치 처리
# =========================

tottenham_df = df.copy()

BATCH_SIZE = 20

all_results = []

comments = tottenham_df["comment_clean"].fillna("").astype(str).tolist()

for start in tqdm(range(0, len(comments), BATCH_SIZE), desc="GPT 배치 감정분석"):
    batch_comments = comments[start:start + BATCH_SIZE]

    batch_results = analyze_sentiment_batch(
        batch_comments,
        start_idx=start
    )

    all_results.extend(batch_results)

    time.sleep(0.2)


# =========================
# 3. 결과 정렬 및 병합
# =========================

result_df = pd.DataFrame(all_results)

result_df = result_df.sort_values("id").reset_index(drop=True)

tottenham_df["gpt_sentiment"] = result_df["sentiment"]
tottenham_df["gpt_confidence"] = result_df["confidence"]
tottenham_df["gpt_target"] = result_df["target"]
tottenham_df["gpt_reason"] = result_df["reason"]


# =========================
# 4. 확인
# =========================

display(
    tottenham_df[
        [
            "comment_clean",
            "gpt_sentiment",
            "gpt_confidence",
            "gpt_target",
            "gpt_reason"
        ]
    ].head()
)


# =========================
# 5. 저장
# =========================

tottenham_df.to_excel(
    "tottenham_gpt_all_labeled.xlsx",
    index=False
)

tottenham_df.to_csv(
    "tottenham_gpt_all_labeled.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 저장 완료")